#  지하철 수요 예측 프로젝트 - 데이터 전처리 및 EDA 테스트

이 노트북은 2024년 서울 지하철 2호선 승하차 데이터와 기상청 **시간대별 날씨 데이터**를 병합하고,
기계학습 및 딥러닝 모델이 학습할 수 있는 **최종 학습 데이터셋**으로 전처리하는 과정을 테스트하는 용도

### 전처리 핵심 목표
1. **2호선 데이터 필터링**: 전체 역 중 2호선 51개 역으로 데이터 압축
2. **승차/하차 개별 Melt**: 승차와 하차 데이터를 나누어 각각 세로로 길게 풀기
3. **시간 변환**: `06시 이전` -> 5시, `24시 이후` -> 0시(실질적인 다음 날 0시)로 매핑
4. **시계열 오프셋 보정**: `24시 이후` (0시로 매핑됨) 데이터의 날짜를 다음 날(D+1)로 1일 자동 보정
5. **승/하차 조인**: 날짜/역명/시간을 기준으로 승차인원과 하차인원을 하나의 행으로 병합
6. **피처 엔지니어링**: 요일(0~6), 월(1~12), 한국 공휴일 여부(0/1) 생성
7. **시간대별 날씨 결합**: 날짜 및 **시간(Hour)**을 기준으로 기온, 강수량, **적설량** 1대1 병합
8. **데이터 타입 최적화**: 메모리 효율 및 모델 학습 속도 극대화를 위해 적절한 타입 다운캐스팅

## 1. 라이브러리 로드

In [46]:
import pandas as pd
import numpy as np
import re
import holidays
import os

## 2. 데이터 불러오기 및 인코딩 확인
- 한국 공공데이터포털 및 기상청 자료는 대개 `cp949` 또는 `euc-kr` 인코딩을 사용합니다.

In [47]:
# 지하철 데이터 로드
subway = pd.read_csv('subway_2023-2024.csv', encoding='utf-8-sig')
print("=== 지하철 데이터 뼈대 ===")
display(subway.head())

# 시간대별 날씨 데이터 로드 (2024_weather_2.csv)
weather = pd.read_csv('weather_2023-2024.csv', encoding='utf-8-sig')
print("=== 시간대별 날씨 데이터 뼈대 ===")
display(weather.head())

=== 지하철 데이터 뼈대 ===


,연번,날짜,호선,역번호,역명,구분,06시 이전,06시-07시,07시-08시,08시-09시,...,15시-16시,16시-17시,17시-18시,18시-19시,19시-20시,20시-21시,21시-22시,22시-23시,23시-24시,24시 이후
0,1,2023-01-01,1호선,150,서울역,승차,215,145,231,594,...,2655,2509,2696,2549,2462,2177,2190,1808,734,7
1,2,2023-01-01,1호선,150,서울역,하차,154,636,595,939,...,2282,2295,2526,1930,1897,1487,991,609,280,46
2,3,2023-01-01,1호선,151,시청,승차,48,73,106,194,...,843,895,959,985,670,630,515,330,146,0
3,4,2023-01-01,1호선,151,시청,하차,64,247,293,463,...,602,575,533,456,285,267,246,154,79,18
4,5,2023-01-01,1호선,152,종각,승차,407,235,158,201,...,1145,1402,1223,1272,911,913,906,602,232,3


=== 시간대별 날씨 데이터 뼈대 ===


,지점,지점명,일시,기온(°C),강수량(mm),적설(cm)
0,108,서울,2023-01-01 01:00,1.5,NaN,NaN
1,108,서울,2023-01-01 02:00,1.5,NaN,NaN
2,108,서울,2023-01-01 03:00,1.6,NaN,NaN
3,108,서울,2023-01-01 04:00,1.5,NaN,NaN
4,108,서울,2023-01-01 05:00,0.8,NaN,NaN


## 3. 2호선 데이터만 필터링
- 데이터양을 압축하고 로컬 학습 속도를 높이기 위해 2호선 노선만 선택합니다.

In [48]:
subway_line2 = subway[subway['호선'] == '2호선'].copy()
print(f"전체 데이터 행 수: {len(subway):,}")
print(f"2호선 데이터 행 수: {len(subway_line2):,}")
print(f"2호선 역 목록 ({subway_line2['역명'].nunique()}개 역):")
print(subway_line2['역명'].unique())

전체 데이터 행 수: 398,680
2호선 데이터 행 수: 73,100
2호선 역 목록 (50개 역):
['시청' '을지로입구' '을지로3가' '을지로4가' '동대문역사문화공원(DDP)' '신당' '상왕십리' '왕십리(성동구청)'
 '한양대' '뚝섬' '성수' '건대입구' '구의(광진구청)' '강변(동서울터미널)' '잠실나루' '잠실(송파구청)' '잠실새내'
 '종합운동장' '삼성(무역센터)' '선릉' '역삼' '강남' '교대(법원.검찰청)' '서초' '방배' '사당' '낙성대(강감찬)'
 '서울대입구(관악구청)' '봉천' '신림' '신대방' '구로디지털단지' '대림(구로구청)' '신도림' '문래' '영등포구청'
 '당산' '합정' '홍대입구' '신촌' '이대' '아현' '충정로(경기대입구)' '용답' '신답' '신설동' '도림천' '양천구청'
 '신정네거리' '용두(동대문구청)']


## 4. 승차 / 하차 데이터 분리 및 개별 Melt
- 한 행에 승차와 하차 인원이 동시에 들어있도록 데이터를 쪼갠 뒤 세로로 풀고(Melt) 다시 결합합니다.

In [49]:
# 1. 승차와 하차 행 분리
subway_in = subway_line2[subway_line2['구분'] == '승차'].copy()
subway_out = subway_line2[subway_line2['구분'] == '하차'].copy()

# 2. 시간대 컬럼 정의
time_columns = [
    '06시 이전', '06시-07시', '07시-08시', '08시-09시', '09시-10시', '10시-11시',
    '11시-12시', '12시-13시', '13시-14시', '14시-15시', '15시-16시',
    '16시-17시', '17시-18시', '18시-19시', '19시-20시', '20시-21시', '21시-22시',
    '22시-23시', '23시-24시', '24시 이후'
]
time_columns = list(dict.fromkeys(time_columns)) # 중복 방지

# 3. 승차 데이터 Melt
in_melted = pd.melt(
    subway_in,
    id_vars=['날짜', '역명', '호선'],
    value_vars=time_columns,
    var_name='시간대',
    value_name='승차인원'
)

# 4. 하차 데이터 Melt
out_melted = pd.melt(
    subway_out,
    id_vars=['날짜', '역명', '호선'],
    value_vars=time_columns,
    var_name='시간대',
    value_name='하차인원'
)

## 5. 시간대 텍스트 변환 및 승하차 데이터 병합 (Merge)
- `06시 이전`은 첫차 기동 시간대인 **5시**로 처리합니다.
- `24시 이후`는 실질적으로 **다음 날 0시**를 뜻하므로 임시 정수 **0**으로 매핑합니다.
- 병합 후 0시(24시 이후)에 해당하는 행들의 날짜를 하루 뒤로 보정(+1일)하여 날씨 데이터와 동기화시킵니다.

In [50]:
def parse_hour(time_str):
    if '06시 이전' in time_str:
        return 5
    elif '24시 이후' in time_str:
        return 0  # 24시 이후는 다음 날 0시이므로 임시로 0으로 매핑
    else:
        match = re.search(r'(\d+)시', time_str)
        return int(match.group(1)) if match else 0

# 시간 정제 함수 적용
in_melted['시간'] = in_melted['시간대'].apply(parse_hour)
out_melted['시간'] = out_melted['시간대'].apply(parse_hour)

# 임시 시간대 컬럼 제거
in_melted = in_melted.drop(columns=['시간대'])
out_melted = out_melted.drop(columns=['시간대'])

# 두 데이터프레임을 날짜, 역명, 호선, 시간 기준으로 병합
subway_melted = pd.merge(
    in_melted,
    out_melted,
    on=['날짜', '역명', '호선', '시간'],
    how='inner'
)

# 날짜 타입 변환
subway_melted['날짜'] = pd.to_datetime(subway_melted['날짜'])

# [핵심 보정] 시간이 0시(원래 24시 이후)인 행들의 날짜를 하루(+1일) 뒤로 보정합니다.
subway_melted.loc[subway_melted['시간'] == 0, '날짜'] = (
    subway_melted.loc[subway_melted['시간'] == 0, '날짜'] + pd.to_timedelta(1, unit='D')
)

print("승하차인원 분리 병합 및 심야 오프셋 보정 완료:")
display(subway_melted.head())

승하차인원 분리 병합 및 심야 오프셋 보정 완료:


,날짜,역명,호선,승차인원,시간,하차인원
0,2023-01-01,시청,2호선,43,5,36
1,2023-01-01,을지로입구,2호선,108,5,114
2,2023-01-01,을지로3가,2호선,52,5,55
3,2023-01-01,을지로4가,2호선,21,5,37
4,2023-01-01,동대문역사문화공원(DDP),2호선,63,5,49


## 6. 날짜 정보 가공 (요일, 월, 한국 공휴일여부)
- 보정된 `날짜`를 기준으로 요일과 월, 그리고 한국 공휴일 정보를 정확히 추출합니다.
- `요일`: 0=월요일, 6=일요일
- `공휴일여부`: 한국 공휴일에 속하면 1, 평일이면 0

In [51]:
# 요일 및 월 추출
subway_melted['요일'] = subway_melted['날짜'].dt.weekday
subway_melted['월'] = subway_melted['날짜'].dt.month

# 한국 공휴일 목록 정의
kr_holidays = holidays.KR()
subway_melted['공휴일여부'] = subway_melted['날짜'].apply(
    lambda x: 1 if x in kr_holidays else 0
)

display(subway_melted.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부
0,2023-01-01,시청,2호선,43,5,36,6,1,1
1,2023-01-01,을지로입구,2호선,108,5,114,6,1,1
2,2023-01-01,을지로3가,2호선,52,5,55,6,1,1
3,2023-01-01,을지로4가,2호선,21,5,37,6,1,1
4,2023-01-01,동대문역사문화공원(DDP),2호선,63,5,49,6,1,1


## 7. 시간대별 날씨 데이터 정제 및 결합
- `2024_weather_2.csv` 데이터에서 `일시`를 정규화하여 `날짜`와 `시간(Hour)`을 각각 분리한 후 1대1 매핑하여 조인합니다.

In [52]:
# 1. 날씨 데이터 일시 형식 변환
weather['일시'] = pd.to_datetime(weather['일시'])

# 2. 일시에서 조인 기준이 될 '날짜' 와 '시간' 피처 추출
weather['날짜'] = weather['일시'].dt.normalize()
weather['시간'] = weather['일시'].dt.hour

# 3. 필요한 기상정보(기온, 강수량, 적설량) 선별 및 컬럼명 재정의
weather_cleaned = weather[['날짜', '시간', '기온(°C)', '강수량(mm)', '적설(cm)']].copy()
weather_cleaned.columns = ['날짜', '시간', '기온', '강수량', '적설']

# 4. 강수량 및 적설 결측치(NaN) ➡️ 0.0 처리
weather_cleaned['강수량'] = weather_cleaned['강수량'].fillna(0.0)
weather_cleaned['적설'] = weather_cleaned['적설'].fillna(0.0)

# 5. 지하철 데이터와 날씨 데이터 조인 (날짜와 시간 기준 1대1 조인)
final_df = pd.merge(subway_melted, weather_cleaned, on=['날짜', '시간'], how='left')

display(final_df.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
0,2023-01-01,시청,2호선,43,5,36,6,1,1,0.8,0.0,0.0
1,2023-01-01,을지로입구,2호선,108,5,114,6,1,1,0.8,0.0,0.0
2,2023-01-01,을지로3가,2호선,52,5,55,6,1,1,0.8,0.0,0.0
3,2023-01-01,을지로4가,2호선,21,5,37,6,1,1,0.8,0.0,0.0
4,2023-01-01,동대문역사문화공원(DDP),2호선,63,5,49,6,1,1,0.8,0.0,0.0


## 8. 최종 데이터 타입 최적화 및 저장
- 최종 피처셋 결측치 존재 여부 및 데이터 통계를 검사합니다.
- 메모리 다이어트 및 모델 학습 효율을 위해 데이터형 축소(Downcasting) 후 저장합니다.

In [53]:
# 1. 문자열 object ➡️ 범주형 category 및 날짜 datetime 변환
final_df['날짜'] = pd.to_datetime(final_df['날짜'])
final_df['역명'] = final_df['역명'].astype('category')
final_df['호선'] = final_df['호선'].astype('category')

# 2. 정수형 컬럼 ➡️ 8비트/32비트 다운캐스팅
int8_cols = ['시간', '요일', '월', '공휴일여부']
for col in int8_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].astype('int8')

if '승차인원' in final_df.columns:
    final_df['승차인원'] = final_df['승차인원'].astype('int32')
    final_df['하차인원'] = final_df['하차인원'].astype('int32')
elif '승하차인원' in final_df.columns:
    final_df['승하차인원'] = final_df['승하차인원'].astype('int32')

# 3. 실수형 기온, 강수량, 적설 ➡️ 32비트 실수 변환
float32_cols = ['기온', '강수량', '적설']
for col in float32_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].astype('float32')

print("=== 최종 데이터셋 타입 정보 ===")
print(final_df.info())

print("\n=== 최종 데이터셋 결측치 검사 ===")
print(final_df.isnull().sum())

print("\n=== 최종 데이터셋 행 개수 ===")
print(f"{len(final_df):,} 행")

print("\n=== 주요 통계 요약 ===")
display(final_df.describe())

# CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig)
output_path = 'final_dataset_230101-241231.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n타입 변환 및 전처리 완료! 데이터가 '{output_path}' 파일로 저장되었습니다.")

=== 최종 데이터셋 타입 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731000 entries, 0 to 730999
Data columns (total 12 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   날짜      731000 non-null  datetime64[ns]
 1   역명      731000 non-null  category      
 2   호선      731000 non-null  category      
 3   승차인원    731000 non-null  int32         
 4   시간      731000 non-null  int8          
 5   하차인원    731000 non-null  int32         
 6   요일      731000 non-null  int8          
 7   월       731000 non-null  int8          
 8   공휴일여부   731000 non-null  int8          
 9   기온      731000 non-null  float32       
 10  강수량     731000 non-null  float32       
 11  적설      731000 non-null  float32       
dtypes: category(2), datetime64[ns](1), float32(3), int32(2), int8(4)
memory usage: 23.7 MB
None

=== 최종 데이터셋 결측치 검사 ===
날짜       0
역명       0
호선       0
승차인원     0
시간       0
하차인원     0
요일       0
월        0
공휴일여부    0
기온       0
강수량   

,날짜,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
count,731000,731000.000000,731000.000000,731000.000000,731000.00000,731000.000000,731000.000000,731000.000000,731000.000000,731000.000000
mean,2024-01-01 01:12:00,1339.236343,13.300000,1359.401101,2.99699,6.519836,0.050616,14.941717,0.173557,0.149856
min,2023-01-01 00:00:00,0.000000,0.000000,0.000000,0.00000,1.000000,0.000000,-17.200001,0.000000,0.000000
25%,2023-07-02 00:00:00,308.000000,8.750000,325.000000,1.00000,4.000000,0.000000,5.400000,0.000000,0.000000
50%,2024-01-01 00:00:00,837.000000,13.500000,815.000000,3.00000,7.000000,0.000000,16.400000,0.000000,0.000000
75%,2024-07-02 00:00:00,1722.000000,18.250000,1737.000000,5.00000,10.000000,0.000000,24.400000,0.000000,0.000000
max,2025-01-01 00:00:00,18966.000000,23.000000,19269.000000,6.00000,12.000000,1.000000,36.299999,39.000000,28.600000
std,NaN,1663.398820,6.148988,1769.592364,2.00328,3.449553,0.219212,10.962975,1.295500,1.227329



타입 변환 및 전처리 완료! 데이터가 'final_dataset_230101-241231.csv' 파일로 저장되었습니다.


In [54]:
final_df.isnull().sum()

날짜       0
역명       0
호선       0
승차인원     0
시간       0
하차인원     0
요일       0
월        0
공휴일여부    0
기온       0
강수량      0
적설       0
dtype: int64

In [55]:
nan_rows = final_df[final_df.isnull().any(axis=1)]
# 결측치 있는 행들을 화면에 출력
display(nan_rows.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설


In [56]:
target_day = final_df[final_df['날짜'] == '2024-05-05']

display(target_day.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
24500,2024-05-05,시청,2호선,22,5,19,6,5,1,17.6,1.7,0.0
24501,2024-05-05,을지로입구,2호선,80,5,80,6,5,1,17.6,1.7,0.0
24502,2024-05-05,을지로3가,2호선,55,5,46,6,5,1,17.6,1.7,0.0
24503,2024-05-05,을지로4가,2호선,24,5,16,6,5,1,17.6,1.7,0.0
24504,2024-05-05,동대문역사문화공원(DDP),2호선,71,5,36,6,5,1,17.6,1.7,0.0


In [57]:
# 조건이 두 개 이상
summer_vacation = final_df[(final_df['날짜'] >= '2024-07-01') & (final_df['날짜'] <= '2024-08-31')]
display(summer_vacation.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
27350,2024-07-01,시청,2호선,33,5,99,0,7,0,22.1,0.0,0.0
27351,2024-07-01,을지로입구,2호선,88,5,92,0,7,0,22.1,0.0,0.0
27352,2024-07-01,을지로3가,2호선,42,5,84,0,7,0,22.1,0.0,0.0
27353,2024-07-01,을지로4가,2호선,28,5,49,0,7,0,22.1,0.0,0.0
27354,2024-07-01,동대문역사문화공원(DDP),2호선,244,5,47,0,7,0,22.1,0.0,0.0


In [58]:
# between(시작일, 종료일).
summer_vacation = final_df[final_df['날짜'].between('2024-07-01', '2024-08-31')]
display(summer_vacation.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
27350,2024-07-01,시청,2호선,33,5,99,0,7,0,22.1,0.0,0.0
27351,2024-07-01,을지로입구,2호선,88,5,92,0,7,0,22.1,0.0,0.0
27352,2024-07-01,을지로3가,2호선,42,5,84,0,7,0,22.1,0.0,0.0
27353,2024-07-01,을지로4가,2호선,28,5,49,0,7,0,22.1,0.0,0.0
27354,2024-07-01,동대문역사문화공원(DDP),2호선,244,5,47,0,7,0,22.1,0.0,0.0


In [59]:
# '월' 컬럼이 12인 행만 뽑아옵니다.
december_data = final_df[final_df['월'] == 12]
display(december_data.head())

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
16700,2023-12-01,시청,2호선,40,5,82,4,12,0,-6.6,0.0,0.0
16701,2023-12-01,을지로입구,2호선,83,5,90,4,12,0,-6.6,0.0,0.0
16702,2023-12-01,을지로3가,2호선,44,5,76,4,12,0,-6.6,0.0,0.0
16703,2023-12-01,을지로4가,2호선,26,5,42,4,12,0,-6.6,0.0,0.0
16704,2023-12-01,동대문역사문화공원(DDP),2호선,202,5,40,4,12,0,-6.6,0.0,0.0


In [60]:
x_data = final_df[final_df['강수량'] == 0]

print("비 안 온 데이터 개수:", len(x_data))

비 안 온 데이터 개수: 682300


In [61]:

# data의 총 행 개수 출력
print("데이터 갯수:", len(final_df))

데이터 갯수: 731000


In [62]:
x_data = final_df[final_df['강수량'] != 0]

print("비 온 데이터 개수:", len(x_data))


비 온 데이터 개수: 48700


In [63]:
final_df.head()

,날짜,역명,호선,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
0,2023-01-01,시청,2호선,43,5,36,6,1,1,0.8,0.0,0.0
1,2023-01-01,을지로입구,2호선,108,5,114,6,1,1,0.8,0.0,0.0
2,2023-01-01,을지로3가,2호선,52,5,55,6,1,1,0.8,0.0,0.0
3,2023-01-01,을지로4가,2호선,21,5,37,6,1,1,0.8,0.0,0.0
4,2023-01-01,동대문역사문화공원(DDP),2호선,63,5,49,6,1,1,0.8,0.0,0.0


In [64]:
final_df.drop(columns=['호선'], inplace=True)
final_df.head()

,날짜,역명,승차인원,시간,하차인원,요일,월,공휴일여부,기온,강수량,적설
0,2023-01-01,시청,43,5,36,6,1,1,0.8,0.0,0.0
1,2023-01-01,을지로입구,108,5,114,6,1,1,0.8,0.0,0.0
2,2023-01-01,을지로3가,52,5,55,6,1,1,0.8,0.0,0.0
3,2023-01-01,을지로4가,21,5,37,6,1,1,0.8,0.0,0.0
4,2023-01-01,동대문역사문화공원(DDP),63,5,49,6,1,1,0.8,0.0,0.0


In [65]:
final_df['승차인원'].mean()

np.float64(1339.236343365253)

In [66]:
final_df['하차인원'].mean()

np.float64(1359.40110123119)